In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))
import json
import time
import pandas as pd
from google import genai
from dotenv import load_dotenv
from collections import defaultdict
from data_loader import load_edos_data
DATA_DIR='../data/'
KG_DIR='../kg/'
RESULTS_DIR='../results/'
os.makedirs(KG_DIR, exist_ok=True) ## Create the KG_DIR if it doesn't exist
load_dotenv('../.env')
GEMINI_API_KEY=os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found in .env file')
client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.5-flash-lite"

c:\Users\Ashwin Nair\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
time.sleep(3)
response = client.models.generate_content( ## Generate a test request to the Gemini API to verify that it is working correctly
    model=GEMINI_MODEL,contents="Reply with OK only.")
print(f'Gemini API working: {response.text.strip()}')

Gemini API working: OK


In [3]:
train_df, _, _ = load_edos_data(DATA_DIR, task='B') ## Load the training data for KG construction (We don't need the dev and test sets for KG construction)
print(f'Sexist training posts for KG construction: {len(train_df)}') ## Print the number of sexist training posts for KG construction
print(f'\nCategory distribution:') ## Print the distribution of categories in the training data
print(train_df['label'].value_counts().to_string())

Sexist training posts for KG construction: 3398

Category distribution:
label
2. derogation                               1590
3. animosity                                1165
4. prejudiced discussions                    333
1. threats, plans to harm and incitement     310


In [4]:
# Fixed relation schema aligned to EDOS Task B categories
# Each relation maps to one or more Task B categories
KG_SCHEMA = { ## Define the relation schema for the knowledge graph
    'STEREOTYPED_AS': '2. derogation', ## Map the relation 'STEREOTYPED_AS' to the category '2. derogation'
    'FRAMED_AS_INFERIOR': '2. derogation', ## Map the relation 'FRAMED_AS_INFERIOR' to the category '2. derogation'
    'ASSIGNED_TO_ROLE': '4. prejudiced discussions', ## Map the relation 'ASSIGNED_TO_ROLE' to the category '4. prejudiced discussions'
    'THREATENED_WITH': '1. threats, plans to harm and incitement', ## Map the relation 'THREATENED_WITH' to the category '1. threats, plans to harm and incitement'
    'EXPRESSED_ANIMOSITY_TOWARDS': '3. animosity', ## Map the relation 'EXPRESSED_ANIMOSITY_TOWARDS' to the category '3. animosity'
    'IDEOLOGICALLY_DISCREDITED': '4. prejudiced discussions', ## Map the relation 'IDEOLOGICALLY_DISCREDITED' to the category '4. prejudiced discussions'
}
VALID_RELATIONS = set(KG_SCHEMA.keys()) ## Create a set of valid relations from the keys of the KG_SCHEMA dictionary
print('KG Relation Schema:')
for relation, category in KG_SCHEMA.items():
    print(f'{relation} - {category}')

KG Relation Schema:
STEREOTYPED_AS - 2. derogation
FRAMED_AS_INFERIOR - 2. derogation
ASSIGNED_TO_ROLE - 4. prejudiced discussions
THREATENED_WITH - 1. threats, plans to harm and incitement
EXPRESSED_ANIMOSITY_TOWARDS - 3. animosity
IDEOLOGICALLY_DISCREDITED - 4. prejudiced discussions


In [5]:
EXTRACTION_PROMPT = """You are a knowledge graph construction expert specialising in sexist language analysis.

Extract semantic triples from the following social media post. Each triple must follow the format:
(subject, RELATION, object)

Only use these exact relation types:
- STEREOTYPED_AS: subject is portrayed through a negative stereotype (maps to derogation)
- FRAMED_AS_INFERIOR: subject is portrayed as less capable or less worthy (maps to derogation)
- ASSIGNED_TO_ROLE: subject is assigned a gender role (maps to prejudiced discussions)
- THREATENED_WITH: subject is threatened with harm (maps to threats)
- EXPRESSED_ANIMOSITY_TOWARDS: subject is the target of hostility or hatred (maps to animosity)
- IDEOLOGICALLY_DISCREDITED: subject's position/role is ideologically dismissed (maps to prejudiced discussions)

Post: "{text}"
Category: {category}

Rules:
- Extract 1 to 3 triples maximum
- Subject and object should be short noun phrases (2-5 words)
- The subject should be the entity being targeted or described (usually women, feminists, girls)
- The object should be the trait, role, threat, or characteristic being assigned to them
- Only extract triples clearly supported by the post
- If no clear triple exists, return NONE

Respond ONLY in this exact format (one triple per line):
(subject, RELATION, object)

Or if no triple exists:
NONE"""

In [6]:
def parse_triples(response_text: str, valid_relations: set) -> list: ### Parse the response text from the LLM to extract triples in the specified format
    triples=[] ## Initialize an empty list to store the extracted triples
    lines=response_text.strip().split('\n') ## Split the response text into lines and iterate over each line
    for line in lines:
        line = line.strip() ## Strip whitespace from the line
        if not line or line.upper() == 'NONE': ## If the line is empty or contains 'NONE', skip to the next line
            continue
        # Remove outer parentheses
        if line.startswith('(') and line.endswith(')'): ## If the line starts and ends with parentheses, remove them
            line = line[1:-1] ## Remove the outer parentheses from the line
        parts = [p.strip() for p in line.split(',')] ## Split the line into parts by commas and strip whitespace from each part
        if len(parts) != 3: ## If the line does not contain exactly 3 parts, skip to the next line
            continue
        subject, relation, obj=parts
        relation=relation.upper().strip()
        # Only keep triples with valid relations
        if relation not in valid_relations:
            # Try partial match
            matched=False ## Initialize a flag to track if a valid relation match is found
            for valid_rel in valid_relations: ## Iterate over the valid relations to find a partial match
                if valid_rel in relation or relation in valid_rel:
                    relation=valid_rel
                    matched=True
                    break
            if not matched: ## If no valid relation match is found, skip to the next line
                continue
        triples.append({
            'subject':  subject.lower().strip(),
            'relation': relation,
            'object':   obj.lower().strip()
        }) ## Append the extracted triple to the list of triples
    return triples

In [ ]:
def extract_triples(text: str, category: str, max_retries: int = 3) -> list: ## Define a function to extract triples from a given text and category, with a maximum number of retries for API calls
    prompt = EXTRACTION_PROMPT.format(text=text, category=category) ## Format the extraction prompt with the provided text and category
    for attempt in range(max_retries): ## Loop to attempt the API call up to max_retries times
        try:
            response = client.models.generate_content(model=GEMINI_MODEL,contents=prompt) ## Generate content using the Gemini API with the specified model and prompt
            if response is None or not hasattr(response, 'text') or response.text is None: ## If the response is None or does not have a 'text' attribute, raise a ValueError
                raise ValueError('Empty response from Gemini')
            raw=response.text.strip()
            triples=parse_triples(raw, VALID_RELATIONS) ## Parse the response text to extract triples using the parse_triples function
            return triples
        except Exception as e:
            error_str = str(e) ## Convert the exception to a string for error handling
            if '429' in error_str or '503' in error_str: ## If the error indicates a rate limit or server overload (HTTP status codes 429 or 503), wait longer before retrying
                # Rate limit or server overload — wait longer
                wait = 60 * (attempt + 1)  # 60s, 120s, 180s
                print(f'Server busy (attempt {attempt+1}/{max_retries}) — waiting {wait}s')
                time.sleep(wait)
            elif attempt < max_retries - 1:
                # Other error — short wait and retry
                print(f'Attempt {attempt+1} failed: {e} — retrying in 15s')
                time.sleep(15)
            else:
                print(f'Failed after {max_retries} attempts: {e}')
    return []

In [ ]:
print('Testing triple extraction on 5 posts:\n')
sample = train_df.sample(5, random_state=42) ## Randomly sample 5 posts from the training data for testing triple extraction
for _, row in sample.iterrows():
    print(f'Post: {row["text"]}') ## Print the text for context
    print(f'Category: {row["label"]}') ## Print the category of the post
    triples=extract_triples(row['text'], row['label']) ## Extract triples from the post text and category using the extract_triples function
    if triples: ## If triples were extracted, print each triple in the specified format
        for t in triples:
            print(f'Triple:({t["subject"]}, {t["relation"]}, {t["object"]})')
    else:
        print('No triples extracted')
    print()
    time.sleep(3)

Testing triple extraction on 5 posts:

Post: ☝️😄 .... 40 years from now she'll complain that she was forced into exploiting her sexuality in order to feed herself and sue MANkind 😂🤣😂 #MAGA #FeminismIsCancer
Category: 4. prejudiced discussions
Triple:(she, ASSIGNED_TO_ROLE, exploiting sexuality)
Triple:(she, ASSIGNED_TO_ROLE, being forced into exploitation)
Triple:(feminism, STEREOTYPED_AS, cancer)

Post: The woman does not even need to be drunk. All she needs to do is regret the sex to make it rape.
Category: 2. derogation
Triple:(the woman, STEREOTYPED_AS, regretful)

Post: This is true. You should only approach a girl without a greenlight if you don't mind being treated like a homeless man at a car dealership.
Category: 2. derogation
Triple:(girl, STEREOTYPED_AS, treated like a homeless man)

Post: Yes, that is so true... But where's the fun in that if I just want to bash the former tranny first lady, Michael? I have to have my priorities.... :)
Category: 3. animosity
Triple:(former 

In [ ]:
def build_kg(df: pd.DataFrame, checkpoint_path: str) -> dict:
    """
    Build the Sexism Knowledge Graph from all training posts.
    
    KG structure:
    {
        "triples": [{"subject": ..., "relation": ..., "object": ...}, ...],
        "entity_index": {"entity": [triple_idx, ...], ...},
        "relation_index": {"RELATION": [triple_idx, ...], ...},
        "stats": {...}
    }
    """
    # Load checkpoint if exists
    if os.path.exists(checkpoint_path): ## If a checkpoint file exists at the specified path, load the checkpoint data
        with open(checkpoint_path, 'r') as f:
            checkpoint=json.load(f)
        all_triples=checkpoint['triples']
        processed=checkpoint['processed']
        print(f'Resuming from checkpoint — {processed} posts already processed')
    else:
        all_triples=[]
        processed=0
    total=len(df)
    failed=0
    no_triple=0
    print(f'Building KG from {total} posts...')
    print(f'Estimated time: ~{total * 3 / 60:.0f} minutes\n')
    for i, (_, row) in enumerate(df.iterrows()): ## Iterate over each row in the DataFrame to process each post for KG construction
        if i < processed: ## If the current index is less than the number of processed posts, skip to the next iteration 
            continue
        if i % 50==0:
            print(f'Progress: {i}/{total} | Triples so far: {len(all_triples)} | Failed: {failed}')
        triples = extract_triples(row['text'], row['label'])
        if triples: ## If triples were extracted, add the source category and post ID to each triple and extend the all_triples list    
            for t in triples:
                t['source_category']=row['label']
                t['post_id']=str(row.get('rewire_id', i))
            all_triples.extend(triples)
        else:
            no_triple += 1
        # Rate limiting
        time.sleep(2)
        # Save checkpoint every 50 posts
        if i%50==0:
            with open(checkpoint_path, 'w') as f:
                json.dump({'triples': all_triples, 'processed': i + 1}, f)
    # Final checkpoint
    with open(checkpoint_path, 'w') as f:
        json.dump({'triples': all_triples, 'processed': total}, f)
    print(f'\nExtraction complete:')
    print(f'Posts processed: {total}')
    print(f'Triples extracted: {len(all_triples)}')
    print(f'Posts with no triple:{no_triple}')
    print(f'Avg triples/post: {len(all_triples)/total:.2f}')
    return all_triples

In [ ]:
def build_indexes(triples: list) -> tuple: ## Define a function to build indexes for fast retrieval of triples based on entities and relations
    entity_index=defaultdict(list) ## Initialize a default dictionary to store the entity index, where each entity maps to a list of triple indices
    relation_index=defaultdict(list) ## Initialize a default dictionary to store the relation index, where each relation maps to a list of triple indices
    for idx, triple in enumerate(triples): ## Iterate over the list of triples, enumerating them to get both the index and the triple
        entity_index[triple['subject']].append(idx) ## Append the index of the triple to the list of indices for the subject entity in the entity index
        entity_index[triple['object']].append(idx) ## Append the index of the triple to the list of indices for the object entity in the entity index
        # Index by relation
        relation_index[triple['relation']].append(idx) ## Append the index of the triple to the list of indices for the relation in the relation index
    return dict(entity_index), dict(relation_index)

In [11]:
checkpoint_path = os.path.join(KG_DIR, 'kg_checkpoint.json') ## Define the path for the checkpoint file to save progress during KG construction
# Build KG
all_triples = build_kg(train_df, checkpoint_path) ## Call the build_kg function to construct the knowledge graph from the training data, passing in the DataFrame and checkpoint path
# Build indexes
entity_index, relation_index = build_indexes(all_triples) ## Call the build_indexes function to create indexes for fast retrieval of triples based on entities and relations
kg = {
    'triples': all_triples,
    'entity_index': entity_index,
    'relation_index':   relation_index,
    'stats': {
        'total_triples': len(all_triples),
        'unique_entities': len(entity_index),
        'unique_relations':len(relation_index),
        'posts_processed': len(train_df)
    }
}
# Save KG
kg_path = os.path.join(KG_DIR, 'sexism_kg.json')
with open(kg_path, 'w') as f:
    json.dump(kg, f, indent=2)
print(f'\nKG saved to {kg_path}')
print(f'\nKG Statistics:')
print(f' Total triples: {kg["stats"]["total_triples"]}')
print(f' Unique entities: {kg["stats"]["unique_entities"]}')
print(f' Unique relations: {kg["stats"]["unique_relations"]}')

Resuming from checkpoint — 3398 posts already processed
Building KG from 3398 posts...
Estimated time: ~170 minutes


Extraction complete:
Posts processed: 3398
Triples extracted: 6270
Posts with no triple:0
Avg triples/post: 1.85

KG saved to ../kg/sexism_kg.json

KG Statistics:
 Total triples: 6270
 Unique entities: 6449
 Unique relations: 6


### KG Analysis

In [12]:
kg_path = os.path.join(KG_DIR, 'sexism_kg.json') ## Define the path to the saved knowledge graph JSON file
with open(kg_path, 'r') as f:
    kg = json.load(f)
all_triples=kg['triples']
entity_index=kg['entity_index']
relation_index=kg['relation_index']
print('Triple count by relation:')
for rel, indices in sorted(relation_index.items()): ## Iterate over the sorted relation index to print the count of triples for each relation
    category = KG_SCHEMA.get(rel, 'unknown')
    print(f' {rel} {len(indices)} triples -> {category}') ## Print the relation, the number of triples associated with it, and the corresponding category from the KG_SCHEMA
print(f'\nTop 20 most common entities:')
entity_counts = {e: len(idxs) for e, idxs in entity_index.items()} ## Create a dictionary to count the number of triples associated with each entity by calculating the length of the list of indices for each entity in the entity index
top_entities=sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)[:20]
for entity, count in top_entities:
    print(f'{entity} {count} triples')
print(f'\nSample triple per relation:')
for rel in VALID_RELATIONS: ## Iterate over the set of valid relations to print a sample triple for each relation
    if rel in relation_index: ## If the relation exists in the relation index, retrieve a sample triple for that relation
        t=all_triples[relation_index[rel][0]]
        print(f'\n{rel}:')
        print(f'({t["subject"]}, {t["relation"]}, {t["object"]})')
        print(f'Category: {t["source_category"]}')

Triple count by relation:
 ASSIGNED_TO_ROLE 760 triples -> 4. prejudiced discussions
 EXPRESSED_ANIMOSITY_TOWARDS 325 triples -> 3. animosity
 FRAMED_AS_INFERIOR 1462 triples -> 2. derogation
 IDEOLOGICALLY_DISCREDITED 50 triples -> 4. prejudiced discussions
 STEREOTYPED_AS 3254 triples -> 2. derogation
 THREATENED_WITH 419 triples -> 1. threats, plans to harm and incitement

Top 20 most common entities:
women 1227 triples
she 569 triples
her 262 triples
woman 188 triples
you 166 triples
bitch 130 triples
girls 105 triples
men 102 triples
girl 56 triples
whore 54 triples
slut 53 triples
a woman 50 triples
females 48 triples
feminists 44 triples
female 44 triples
cunt 42 triples
wife 41 triples
bitches 40 triples
he 37 triples
they 36 triples

Sample triple per relation:

FRAMED_AS_INFERIOR:
(asian ladies, FRAMED_AS_INFERIOR, 5/10)
Category: 2. derogation

THREATENED_WITH:
(her, THREATENED_WITH, touch boobs)
Category: 1. threats, plans to harm and incitement

IDEOLOGICALLY_DISCREDITED:


In [ ]:
alignment_counts = defaultdict(lambda: defaultdict(int)) ## Initialize a nested default dictionary to count the alignment of relations to their expected categories
for triple in all_triples:
    rel = triple['relation']
    cat = triple['source_category']
    expected = KG_SCHEMA.get(rel, 'unknown')
    alignment_counts[rel][cat] += 1

for rel, cat_counts in alignment_counts.items(): ## Iterate over the alignment counts to print the alignment of relations to their expected categories
    expected=KG_SCHEMA[rel]
    total=sum(cat_counts.values()) 
    correct=cat_counts.get(expected, 0)
    pct= correct / total * 100 if total > 0 else 0
    print(f'{rel}:')
    print(f'Expected: {expected}')
    print(f'Alignment: {correct}/{total} ({pct:.1f}%)')
    for cat, count in sorted(cat_counts.items(), key=lambda x: x[1], reverse=True):
        marker = 'O' if cat == expected else 'X'
        print(f' {marker} {cat}: {count}')
    print()

STEREOTYPED_AS:
 Expected: 2. derogation
Alignment: 1809/3254 (55.6%)
 O 2. derogation: 1809
 X 3. animosity: 1105
 X 4. prejudiced discussions: 229
 X 1. threats, plans to harm and incitement: 111

FRAMED_AS_INFERIOR:
 Expected: 2. derogation
Alignment: 855/1462 (58.5%)
 O 2. derogation: 855
 X 3. animosity: 374
 X 4. prejudiced discussions: 182
 X 1. threats, plans to harm and incitement: 51

ASSIGNED_TO_ROLE:
 Expected: 4. prejudiced discussions
Alignment: 192/760 (25.3%)
 X 3. animosity: 275
 X 2. derogation: 244
 O 4. prejudiced discussions: 192
 X 1. threats, plans to harm and incitement: 49

THREATENED_WITH:
 Expected: 1. threats, plans to harm and incitement
Alignment: 296/419 (70.6%)
 O 1. threats, plans to harm and incitement: 296
 X 3. animosity: 60
 X 2. derogation: 55
 X 4. prejudiced discussions: 8

EXPRESSED_ANIMOSITY_TOWARDS:
 Expected: 3. animosity
Alignment: 230/325 (70.8%)
 O 3. animosity: 230
 X 2. derogation: 81
 X 4. prejudiced discussions: 7
 X 1. threats, plans 

In [ ]:
print('KG Cleaning\n')

# Step 1 — Entity normalization mapping
ENTITY_NORMALIZER = {
    'she':      'women',
    'her':      'women',
    'woman':    'women',
    'a woman':  'women',
    'female':   'women',
    'females':  'women',
    'girl':     'women',
    'girls':    'women',
    'he':       'men',
    'him':      'men',
}

# Step 2 — Pronouns to filter out entirely (too ambiguous to normalize)
PRONOUNS_TO_DROP = {'they', 'you', 'it', 'we', 'i', 'this', 'that', 'them', 'their'}

# Step 3 — Relations to exclude (ASSIGNED_TO_ROLE: 25.3% alignment — worse than random)
RELATIONS_TO_DROP = {'ASSIGNED_TO_ROLE'}

def normalize_entity(entity: str) -> str: ## Define a function to normalize entity names by mapping pronouns and variants to canonical entity names
    """Normalize pronouns and variants to canonical entity names."""
    e = entity.lower().strip()
    return ENTITY_NORMALIZER.get(e, e)

def is_valid_entity(entity: str) -> bool: ### Define a function to check if an entity is valid (not an ambiguous pronoun)
    """Return False if entity is an ambiguous pronoun that should be dropped."""
    return entity.lower().strip() not in PRONOUNS_TO_DROP

def clean_kg(all_triples: list) -> list: ### Define a function to clean the knowledge graph by normalizing entities, dropping ambiguous pronouns, and removing unreliable relations
    """
    Clean the KG by:
    1. Normalizing pronouns (she/her → women, he/him → men)
    2. Dropping ambiguous pronoun entities (they, you, it, etc.)
    3. Removing unreliable relations (ASSIGNED_TO_ROLE)
    """
    cleaned = []
    stats = {
        'original': len(all_triples),
        'dropped_relation': 0,
        'dropped_pronoun': 0,
        'normalized': 0,
        'kept': 0
    }

    for triple in all_triples:
        # Drop unreliable relations
        if triple['relation'] in RELATIONS_TO_DROP:
            stats['dropped_relation'] += 1
            continue

        # Normalize entities
        subj = normalize_entity(triple['subject'])
        obj  = normalize_entity(triple['object'])

        if subj != triple['subject'] or obj != triple['object']:
            stats['normalized'] += 1

        # Drop triples where subject or object is ambiguous pronoun
        if not is_valid_entity(subj) or not is_valid_entity(obj):
            stats['dropped_pronoun'] += 1
            continue

        cleaned.append({
            'subject': subj,
            'relation': triple['relation'],
            'object': obj,
            'source_category': triple['source_category'],
            'post_id': triple['post_id']
        })
        stats['kept'] += 1

    return cleaned, stats

# Run cleaning
clean_triples, stats = clean_kg(all_triples)

print(f'Cleaning results:')
print(f'Original triples:  {stats["original"]}')
print(f'Dropped (bad relation): {stats["dropped_relation"]}')
print(f'Dropped (ambiguous pronoun): {stats["dropped_pronoun"]}')
print(f'Normalized (she→women etc): {stats["normalized"]}')
print(f'Final clean triples: {stats["kept"]}')

# Rebuild indexes on clean triples
clean_entity_index, clean_relation_index = build_indexes(clean_triples)

print(f'\nClean KG entity stats:')
clean_entity_counts = {e: len(idxs) for e, idxs in clean_entity_index.items()}
top_clean = sorted(clean_entity_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for entity, count in top_clean:
    print(f'  {entity} {count} triples')

print(f'\nClean KG relation stats:')
for rel, indices in sorted(clean_relation_index.items()):
    print(f'  {rel} {len(indices)} triples')

# Save clean KG
clean_kg_path = os.path.join(KG_DIR, 'sexism_kg_clean.json')
clean_kg_obj = {
    'triples': clean_triples,
    'entity_index': clean_entity_index,
    'relation_index': clean_relation_index,
    'stats': {
        'total_triples': len(clean_triples),
        'unique_entities': len(clean_entity_index),
        'unique_relations': len(clean_relation_index),
        'cleaning_stats': stats
    }
}
with open(clean_kg_path, 'w') as f:
    json.dump(clean_kg_obj, f, indent=2)

print(f'\nClean KG saved to {clean_kg_path}')

=== KG Cleaning ===

Cleaning results:
  Original triples:  6270
  Dropped (bad relation): 760
  Dropped (ambiguous pronoun): 224
  Normalized (she→women etc): 1187
  Final clean triples: 5286

Clean KG entity stats:
  women                          2173 triples
  men                            110 triples
  bitch                          109 triples
  slut                           51 triples
  whore                          45 triples
  feminists                      41 triples
  wife                           32 triples
  bitches                        32 triples
  cunt                           32 triples
  this bitch                     32 triples

Clean KG relation stats:
  EXPRESSED_ANIMOSITY_TOWARDS         263 triples
  FRAMED_AS_INFERIOR                  1423 triples
  IDEOLOGICALLY_DISCREDITED           49 triples
  STEREOTYPED_AS                      3152 triples
  THREATENED_WITH                     399 triples

Clean KG saved to ../kg/sexism_kg_clean.json
